In [ ]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patches import Patch
from scipy.signal import savgol_filter
import os

from unityvr.preproc import logproc
from unityvr.viz import viz

#from sys import path
from os.path import sep, exists
from os import mkdir, makedirs, getcwd

# import pycircstat

In [ ]:
def run_analysis_for_trial(rootDir, flyID, trial_folder):
    
    ####################################################################################################
    #-----------------------------------------------------------------------------------------------#
    # Define directories
    #-----------------------------------------------------------------------------------------------#
    ####################################################################################################
    dataDir = os.path.join(rootDir, 'raw', flyID, trial_folder)
    preprocDir = os.path.join(rootDir, 'preproc', flyID, trial_folder)
    plotDir = os.path.join(rootDir, 'plot', flyID, trial_folder)

    os.makedirs(preprocDir, exist_ok=True)
    os.makedirs(plotDir, exist_ok=True)
    ####################################################################################################
    #-----------------------------------------------------------------------------------------------#
    #Load data
    #-----------------------------------------------------------------------------------------------#
    ####################################################################################################
    # Load JSON
    json_files = [f for f in os.listdir(dataDir) if f.endswith('.json')]
    if not json_files:
        print(f"⚠️ No JSON file in {dataDir}, skipping.")
        return

    fileName = json_files[0]
    dat = logproc.openUnityLog(dataDir, fileName)

    logproc.makeMetaDict(dat, fileName)
    objDf = logproc.objDfFromLog(dat)
    posDf, ftDf, tsDf = logproc.timeseriesDfFromLog(dat) 

    # Extract ball radius
    matching = [s for s in dat if "ficTracBallRadius" in s] 
    ballRadius = matching[0]["ficTracBallRadius"]
    dc2cm = 10
    
    ####################################################################################################
    #-----------------------------------------------------------------------------------------------#
    #Import stimulus data
    #-----------------------------------------------------------------------------------------------#
    ####################################################################################################
    
    # Import stimulus angle data
    ## All decimal points (more precise for stimulus speed measurements)
    from decimal import Decimal, getcontext, ROUND_HALF_UP
    getcontext().prec = 50  # enough precision (>20 dp)

    # Read as strings first so nothing is rounded
    df = pd.read_csv('/Volumes/otopaliklab/Temporary_Storage/Aisha/UnityProtocols/new2_fh_3.1_6.5/darkspot-20degspot-fh-125degpersec-120fps-anglerange90deg--panWidth_px-1504-panHeight_px-582/imageDf.csv', dtype=str)

    # Convert selected columns to Decimal (exact)
    for col in ['time [s]', 'angular_shift (deg)']:
        df[col] = df[col].map(Decimal)
        
    # Show full precision (Decimal prints all digits)
    print(df.head().to_string(index=False))
    
    
    